# Road Survey AI — Training di Google Colab (GPU T4 gratis)

Pakai notebook ini bila laptop **tanpa GPU NVIDIA**, atau mau training cepat di GPU gratis.

**Langkah:** aktifkan GPU (`Runtime > Change runtime type > T4 GPU`), lalu jalankan sel berurutan.

Output `best.pt` diunduh ke laptop, lalu dipakai:
`python road_survey.py --source clip.mp4 --model best.pt`

In [ ]:
# 1. Cek GPU
!nvidia-smi
!pip -q install ultralytics

In [ ]:
import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

## 2. Siapkan dataset
Dua pilihan:
- **A.** Upload dataset YOLO (hasil `voc_to_yolo.py`) ke Google Drive lalu mount.
- **B.** Unduh RDD2022 langsung di Colab (cepat, internet Colab kencang).

In [ ]:
# Pilihan A: mount Google Drive (dataset sudah diupload ke sana)
from google.colab import drive
drive.mount('/content/drive')
# Sesuaikan path data.yaml kamu, mis:
DATA = '/content/drive/MyDrive/road_survey/datasets/rdd_yolo/data.yaml'

In [ ]:
# Pilihan B: unduh RDD2022 + konversi langsung di Colab
# (clone repo ini agar dapat script-nya, atau upload scripts/ manual)
# !git clone <URL_REPO_ANDA> rs && cd rs
# !python scripts/download_rdd2022.py --country Japan India --out datasets/RDD2022
# !python scripts/voc_to_yolo.py --input datasets/RDD2022/Japan datasets/RDD2022/India --out datasets/rdd_yolo
# DATA = 'datasets/rdd_yolo/data.yaml'

In [ ]:
# 3. Training
from ultralytics import YOLO
model = YOLO('yolov8s.pt')   # ganti yolov8m.pt bila ingin lebih akurat
model.train(data=DATA, epochs=100, imgsz=640, batch=-1, device=0,
            name='road_damage', patience=30,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, mosaic=1.0)

In [ ]:
# 4. Unduh bobot terbaik ke laptop
from google.colab import files
files.download('runs/detect/road_damage/weights/best.pt')

## 5. Fine-tune data lokal Luwu Timur
Ulangi sel training dengan `YOLO('runs/detect/road_damage/weights/best.pt')` sebagai
titik awal dan `DATA` menunjuk ke dataset lokal kamu (lihat `docs/LABELING.md`).